# Day 19: Handling RAG Context Overflow with Sliding Windows

## Core Theory (Just-in-Time)
When building Retrieval-Augmented Generation (RAG) systems, a common issue is **Context Overflow**. Large Language Models (LLMs) have a maximum context window limit (e.g., 4k, 8k, 128k tokens). If your combined retrieved documents exceed this limit, the API call will fail or the model will forget the beginning of the context (a phenomenon known as "Lost in the Middle").

To solve this natively before passing the prompt to the LLM, we can implement a **Sliding Window** approach over the retrieved text. Instead of passing all documents as one giant string, we slice the text into manageable, overlapping chunks, or simply truncate the retrieved documents list until it fits within our context budget. Understanding how to manage this in standard Python ensures you are not overly reliant on "black box" abstractions.

### AI Security Implications
When dealing with retrieved text from a database, it's crucial to consider security:
- **PII Protection:** Truncating text can accidentally slice identifiers (like cutting a credit card number in half). Ensure PII scrubbing happens *before* or *during* the retrieval phase, prior to truncation.
- **Prompt Injection:** Attackers might hide malicious instructions deep in a document. If truncation leaves *only* the malicious instruction and cuts out the context, the LLM might execute it. Sliding windows ensure context is maintained.
- **Fallbacks:** Always implement a fallback mechanism (e.g., returning a default 'I cannot answer this' message) if the retrieved context is completely empty after safety filters are applied.

In [1]:
from typing import List

def truncate_context_sliding_window(retrieved_texts: List[str], max_words: int = 100) -> str:
    """
    Combines retrieved texts and truncates them to a maximum word count to prevent context overflow.
    In production, you would swap the word count for an exact token count (e.g., via tiktoken).
    
    Args:
        retrieved_texts: A list of document strings.
        max_words: Maximum allowed words for the combined context (approximation for tokens).
        
    Returns:
        A single string containing the combined and safely truncated context.
    """
    combined_words: List[str] = []
    
    for text in retrieved_texts:
        words = text.split()
        if len(combined_words) + len(words) > max_words:
            # Slice the words to fit exactly within the remaining allowance
            allowance = max_words - len(combined_words)
            combined_words.extend(words[:allowance])
            break  # Context window is full
        
        combined_words.extend(words)
        
    return " ".join(combined_words)


In [2]:
# Example Usage:
retrieved_docs = [
    "This is the first highly relevant document that we retrieved from Qdrant.",
    "This is the second document with more specific details about the architecture.",
    "This is a third document that might be too long to fit into our small simulated window."
]

# Set a deliberately small max_words to demonstrate truncation
safe_context = truncate_context_sliding_window(retrieved_docs, max_words=20)
print("--- Safe Context ---")
print(safe_context)


--- Safe Context ---
This is the first highly relevant document that we retrieved from Qdrant. This is the second document with more specific


## Medium Implementation: Object-Oriented Sliding Window
This implementation introduces clean OOP and state management to handle document streams.

In [3]:
from typing import List, Optional

class ContextManager:
    """Manages retrieved context using a sliding window to prevent overflow."""
    def __init__(self, max_words: int):
        self.max_words = max_words
        self._current_words: List[str] = []
        
    def add_document(self, text: str) -> bool:
        """
        Adds a document to the context if there is space.
        Returns True if fully added, False if truncated or ignored.
        """
        words = text.split()
        remaining_space = self.max_words - len(self._current_words)
        
        if remaining_space <= 0:
            return False
            
        if len(words) > remaining_space:
            self._current_words.extend(words[:remaining_space])
            return False
            
        self._current_words.extend(words)
        return True
        
    def get_context(self) -> str:
        return " ".join(self._current_words)

    def clear(self) -> None:
        self._current_words.clear()

# Usage
manager = ContextManager(max_words=20)
manager.add_document("The quick brown fox jumps over the lazy dog.")
manager.add_document("A second document with some more words that might exceed the limit.")
print("Medium Output:", manager.get_context())


Medium Output: The quick brown fox jumps over the lazy dog. A second document with some more words that might exceed the


## Advanced Implementation: Production-Grade Sliding Window
This version uses strict type hinting, docstrings, error handling, and exact imports to prepare for a robust production environment.

In [4]:
import logging
from typing import List, Sequence
from pydantic import BaseModel, Field, ValidationError

# Configure basic logging for production visibility
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class Document(BaseModel):
    """Represents a retrieved document with metadata."""
    content: str = Field(..., min_length=1)
    source_id: str

class SlidingWindowContextBuilder:
    """
    Builds a prompt context from multiple documents using a word-based sliding window.
    In a real system, you would use token counting (e.g., tiktoken).
    """
    
    def __init__(self, max_tokens: int):
        if max_tokens <= 0:
            raise ValueError("max_tokens must be greater than 0")
        self.max_tokens = max_tokens
        self._buffer: List[str] = []
        
    def build_context(self, documents: Sequence[Document]) -> str:
        """
        Iterates through documents, appending their content until the maximum limit is reached.
        """
        self._buffer.clear()
        current_token_count = 0
        
        for doc in documents:
            words = doc.content.split()
            if not words:
                continue
                
            if current_token_count + len(words) > self.max_tokens:
                allowance = self.max_tokens - current_token_count
                if allowance > 0:
                    self._buffer.extend(words[:allowance])
                    logger.warning(f"Document {doc.source_id} was truncated to fit context.")
                else:
                    logger.warning(f"Document {doc.source_id} was entirely skipped due to full context.")
                break  # Context is full
                
            self._buffer.extend(words)
            current_token_count += len(words)
            
        if not self._buffer:
            logger.error("Context is empty. Applying fallback mechanism.")
            return "No relevant context found."
            
        return " ".join(self._buffer)

# Usage Example
try:
    docs = [
        Document(content="This is the first highly relevant document.", source_id="doc_1"),
        Document(content="Here is the second document with additional details.", source_id="doc_2")
    ]
    
    builder = SlidingWindowContextBuilder(max_tokens=12)
    final_context = builder.build_context(docs)
    print("Advanced Output:", final_context)
except ValidationError as e:
    logger.error(f"Invalid document data: {e}")
except Exception as e:
    logger.exception(f"An unexpected error occurred: {e}")


Advanced Output: This is the first highly relevant document. Here is the second document


## Common Pitfalls in Production
1. **Blind Truncation:** Simply chopping off text at an arbitrary character count often splits words or mid-sentence, leading to corrupted context and LLM hallucinations. Always use token-based or word-based truncation.
2. **Lost in the Middle:** If you pack the maximum context window full of retrieved documents, LLMs tend to ignore information in the middle. Often, it is better to strictly prioritize and send *fewer* high-quality tokens rather than filling the entire available context window.
3. **Tokenizer Mismatch:** Using an outdated or mismatched tokenizer (e.g., using a BERT tokenizer for an OpenAI model) will result in inaccurate token counts, causing context overflow errors during the actual API call. While we used words above for demonstration, production systems require model-specific tokenizers.

## Practical Lab / Homework
**Your Task:** 
Implement a function `sliding_window_chunks` that takes a long single document and returns overlapping chunks of it, ensuring no single chunk exceeds `window_size_words`, and consecutive chunks overlap by `overlap_words`.

**Video Walkthrough:** Record a brief async video (2-3 minutes) explaining your design decisions for the chunking logic, how it manages state, and any edge cases you considered.

In [5]:
from typing import List

def sliding_window_chunks(document: str, window_size_words: int = 15, overlap_words: int = 5) -> List[str]:
    """
    Splits a document into overlapping word-based chunks.
    
    Args:
        document: The large input text to chunk.
        window_size_words: The maximum number of words per chunk.
        overlap_words: The number of overlapping words between consecutive chunks.
        
    Returns:
        A list of string chunks.
    """
    words = document.split()
    chunks: List[str] = []
    start = 0
    total_words = len(words)
    
    while start < total_words:
        end = start + window_size_words
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))
        
        if end >= total_words:
            break
            
        start += (window_size_words - overlap_words)
        
    return chunks

# Lab Verification
long_document = "AI Engineering requires understanding how to effectively manage context limits in large language models properly. " * 3
chunks = sliding_window_chunks(long_document, window_size_words=10, overlap_words=3)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {repr(chunk)}")


Chunk 1: 'AI Engineering requires understanding how to effectively manage context limits'
Chunk 2: 'manage context limits in large language models properly. AI Engineering'
Chunk 3: 'properly. AI Engineering requires understanding how to effectively manage context'
Chunk 4: 'effectively manage context limits in large language models properly. AI'
Chunk 5: 'models properly. AI Engineering requires understanding how to effectively manage'
Chunk 6: 'to effectively manage context limits in large language models properly.'


## Reference Links
- [LangChain: Text Splitters & Chunking](https://python.langchain.com/docs/modules/data_connection/document_transformers/)
- [OpenAI Cookbook: Chunking Strategies](https://github.com/openai/openai-cookbook)
- [Qdrant: Handling Long Texts in Vector Search](https://qdrant.tech/articles/handling-long-texts/)